In [35]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Test on split data

In [18]:
df_final_train = pd.read_csv(r'c:\mydata\G8Vitamin\data\final\24092025_baseline\processed_train.csv')
df_final_test  = pd.read_csv(r'c:\mydata\G8Vitamin\data\final\24092025_baseline\processed_test.csv')


In [19]:
threshold = 1e-10

# Select numeric columns
numeric_cols = df_final_train.select_dtypes(include='number').columns

# Apply threshold replacement only on numeric columns
df_final_train[numeric_cols] = df_final_train[numeric_cols].applymap(
    lambda x: 0 if abs(x) < threshold else x
)
df_final_test[numeric_cols] = df_final_test[numeric_cols].applymap(
    lambda x: 0 if abs(x) < threshold else x
)


C:\Users\iseT1enLoc\AppData\Local\Temp\ipykernel_29524\2484542337.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_final_train[numeric_cols] = df_final_train[numeric_cols].applymap(
C:\Users\iseT1enLoc\AppData\Local\Temp\ipykernel_29524\2484542337.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_final_test[numeric_cols] = df_final_test[numeric_cols].applymap(


In [20]:
df_final_train['SmokeFam'] = df_final_train.apply(
    lambda row: 0 if row['SmokeFam'] == 2 and int(row['YearStart']) <= 2011 else 1,
    axis=1
)

In [21]:
df_final_test['SmokeFam'] = df_final_test.apply(
    lambda row: 0 if row['SmokeFam'] == 0 else 1,
    axis=1
)

In [22]:
df_final_train['SmokeFam'].value_counts()

SmokeFam
0    28669
1     7895
Name: count, dtype: int64

In [23]:
df_final_test['SmokeFam'].value_counts()

SmokeFam
0    4559
1    2213
Name: count, dtype: int64

In [24]:
columns_remove = [
    'VitaminD',
    'SEQN',
    'YearID',
    'YearStart',
]
df_final_train.drop(columns=columns_remove,inplace=True)
df_final_test.drop(columns=columns_remove,inplace=True)

In [25]:
df_final_train = df_final_train[df_final_train['milk_consumption']<=3]
df_final_test = df_final_test[df_final_test['milk_consumption']<=3]

In [26]:
df_final_train['SmokeFam'].value_counts()

SmokeFam
0    28555
1     7853
Name: count, dtype: int64

In [27]:
df_final_test['SmokeFam'].value_counts()

SmokeFam
0    4542
1    2207
Name: count, dtype: int64

In [28]:
df_final_train.to_csv(r'c:\mydata\G8Vitamin\data\final\24092025_baseline\processed_train_v2.csv',index=False)
df_final_test.to_csv(r'c:\mydata\G8Vitamin\data\final\24092025_baseline\processed_test_v2.csv',index=False)

In [29]:
import pandas as pd
df_train = pd.read_csv(r'C:\mydata\G8Vitamin\data\final\24092025_baseline\processed_train_v2.csv')
df_test = pd.read_csv(r'C:\mydata\G8Vitamin\data\final\24092025_baseline\processed_test_v2.csv')

In [30]:
df_train['label'].value_counts()

label
0.0    22806
1.0    13602
Name: count, dtype: int64

In [31]:
df_test['label'].value_counts()

label
0.0    4581
1.0    2168
Name: count, dtype: int64

In [32]:
df_final_train.columns

Index(['Gender', 'Age', 'Race', 'familysize', 'PIR', 'BMI', 'Hba1c',
       'milk_consumption', 'SmokeFam', 'label'],
      dtype='object')

In [33]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    brier_score_loss
)
from sklearn.metrics import precision_recall_fscore_support
# === 1. Features and Label ===
X_train = df_final_train.drop(columns=["label"])
y_train = df_final_train["label"].values
X_test  = df_final_test.drop(columns=["label"])
y_test  = df_final_test["label"].values

# === 2. Define categorical and numerical features ===
categorical_features = ["Gender", "Race", "milk_consumption", "SmokeFam"]
numerical_features = [col for col in X_train.columns if col not in categorical_features]

# === 3. Preprocessing ===
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

# === 4. Models ===
models = {
    "GBM": GradientBoostingClassifier(),
    "LR": LogisticRegression(max_iter=1000),
    "Nnet": MLPClassifier(max_iter=1000),
    "RF": RandomForestClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss")
}

# === 5. Evaluation Function (no random split) ===
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, y_pred, labels=[0, 1], zero_division=0
    )
    results = {
        "AUC": roc_auc_score(y_test, y_proba),
        "ACC": accuracy_score(y_test, y_pred),
        "PPV": precision_score(y_test, y_pred, zero_division=0),
        "NPV": tn / (tn + fn) if (tn + fn) > 0 else 0,
        "SEN": recall_score(y_test, y_pred),  # Sensitivity (Recall for 1)
        "SPE": tn / (tn + fp) if (tn + fp) > 0 else 0,  # Specificity (Recall for 0)
        "F1_binary": f1_score(y_test, y_pred, average="binary"),
        "F1_macro": f1_score(y_test, y_pred, average="macro"),
        "F1_weighted": f1_score(y_test, y_pred, average="weighted"),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "KAPPA": cohen_kappa_score(y_test, y_pred),
        "Brier score": brier_score_loss(y_test, y_proba),
        
        "Precision_0": precision[0],
        "Recall_0": recall[0],
        "F1_0": f1[0],
        "Support_0": support[0],
        "Precision_1": precision[1],
        "Recall_1": recall[1],
        "F1_1": f1[1],
        "Support_1": support[1],
    }

    return results

# === 6. Run All Models ===
results = {}
for name, model in models.items():
    print(f"Evaluating {name}...")
    results[name] = evaluate_model(name, model, X_train, y_train, X_test, y_test)

# === 7. Export Results ===
results_df = pd.DataFrame(results).T
results_df.index.name = "Method"
results_df.to_csv(r"C:\mydata\G8Vitamin\src\models\baselineq1\24092025_rerun_paper\evaluation_results.csv")
print(results_df)


Evaluating GBM...
Evaluating LR...
Evaluating Nnet...
Evaluating RF...
Evaluating XGBoost...
              AUC       ACC       PPV       NPV       SEN       SPE  \
Method                                                                
GBM      0.750807  0.718180  0.557476  0.802029  0.595018  0.776468   
LR       0.738909  0.704549  0.535832  0.799352  0.600092  0.753984   
Nnet     0.735324  0.706623  0.540517  0.793633  0.578413  0.767300   
RF       0.732199  0.703067  0.534541  0.794514  0.585332  0.758786   
XGBoost  0.741726  0.710476  0.544845  0.801054  0.599631  0.762934   

         F1_binary  F1_macro  F1_weighted       MCC     KAPPA  Brier score  \
Method                                                                       
GBM       0.575636  0.682339     0.720489  0.365447  0.365013     0.189438   
LR        0.566144  0.671075     0.708591  0.344501  0.343235     0.196238   
Nnet      0.558824  0.669534     0.709117  0.339882  0.339445     0.195020   
RF        0.558785 

c:\mydata\G8Vitamin\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:36:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


# SMOTE

In [34]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    brier_score_loss
)
from imblearn.over_sampling import SMOTE

# === 1. Features and Label ===
X_train = df_final_train.drop(columns=["label"])
y_train = df_final_train["label"].values
X_test  = df_final_test.drop(columns=["label"])
y_test  = df_final_test["label"].values

# === 2. Define categorical and numerical features ===
categorical_features = ["Gender", "Race", "milk_consumption", "SmokeFam"]
numerical_features = [col for col in X_train.columns if col not in categorical_features]

# === 3. Preprocessing ===
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

# === 4. Models ===
models = {
    "GBM": GradientBoostingClassifier(),
    "LR": LogisticRegression(max_iter=1000),
    "Nnet": MLPClassifier(max_iter=1000),
    "RF": RandomForestClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss")
}

# === 5. Evaluation Function (with SMOTE on training data only) ===
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    # 1. Apply preprocessing first
    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)

    # 2. Apply SMOTE on processed training data
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_processed, y_train)

    # 3. Train model
    model.fit(X_train_res, y_train_res)

    # 4. Predict
    y_pred = model.predict(X_test_processed)
    y_proba = model.predict_proba(X_test_processed)[:, 1]

    # 5. Metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, y_pred, labels=[0, 1], zero_division=0
    )
    results = {
        "AUC": roc_auc_score(y_test, y_proba),
        "ACC": accuracy_score(y_test, y_pred),
        "PPV": precision_score(y_test, y_pred, zero_division=0),
        "NPV": tn / (tn + fn) if (tn + fn) > 0 else 0,
        "SEN": recall_score(y_test, y_pred),  # Sensitivity (Recall for 1)
        "SPE": tn / (tn + fp) if (tn + fp) > 0 else 0,  # Specificity
        "F1_binary": f1_score(y_test, y_pred, average="binary"),
        "F1_macro": f1_score(y_test, y_pred, average="macro"),
        "F1_weighted": f1_score(y_test, y_pred, average="weighted"),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "KAPPA": cohen_kappa_score(y_test, y_pred),
        "Brier score": brier_score_loss(y_test, y_proba),

        "Precision_0": precision[0],
        "Recall_0": recall[0],
        "F1_0": f1[0],
        "Support_0": support[0],
        "Precision_1": precision[1],
        "Recall_1": recall[1],
        "F1_1": f1[1],
        "Support_1": support[1],        
    }

    return results

# === 6. Run All Models ===
results = {}
for name, model in models.items():
    print(f"Evaluating {name} with SMOTE...")
    results[name] = evaluate_model(name, model, X_train, y_train, X_test, y_test)

# === 7. Export Results ===
results_df = pd.DataFrame(results).T
results_df.index.name = "Method"
results_df.to_csv(r"C:\mydata\G8Vitamin\src\models\baselineq1\24092025_rerun_paper\evaluation_results_smote.csv")
print(results_df)


Evaluating GBM with SMOTE...
Evaluating LR with SMOTE...
Evaluating Nnet with SMOTE...
Evaluating RF with SMOTE...
Evaluating XGBoost with SMOTE...


c:\mydata\G8Vitamin\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:40:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              AUC       ACC       PPV       NPV       SEN       SPE  \
Method                                                                
GBM      0.747609  0.667062  0.487790  0.832100  0.727860  0.638289   
LR       0.736618  0.639206  0.463071  0.842374  0.772140  0.576293   
Nnet     0.728687  0.647948  0.468788  0.822651  0.720480  0.613621   
RF       0.730056  0.679360  0.500710  0.807428  0.650830  0.692862   
XGBoost  0.736313  0.696399  0.522177  0.811363  0.646218  0.720148   

         F1_binary  F1_macro  F1_weighted       MCC     KAPPA  Brier score  \
Method                                                                       
GBM       0.584120  0.653271     0.677994  0.342238  0.324134     0.208945   
LR        0.578938  0.631660     0.650509  0.326232  0.296342     0.229215   
Nnet      0.568000  0.635463     0.659583  0.312042  0.292711     0.225275   
RF        0.565985  0.655878     0.688018  0.325430  0.318539     0.208537   
XGBoost   0.577613  0.670326     0

In [38]:
def is_diabetes(row):
    if row >= 6.5:
        return 1
    else:
        return 0

In [39]:
df_final_train = pd.read_csv(r'c:\mydata\G8Vitamin\data\final\24092025_baseline\processed_train_v2.csv')
df_final_test  = pd.read_csv(r'c:\mydata\G8Vitamin\data\final\24092025_baseline\processed_test_v2.csv')


In [ ]:
df_final_train['Hba1c'] = df_final_train['Hba1c'].apply(is_diabetes)
df_final_test['Hba1c'] = df_final_test['Hba1c'].apply(is_diabetes)

NameError: name 'isDiabete' is not defined

In [143]:
#diebete assign:
def isDiabete(row):
    if row<6.5:
        return 0
    else:
        return 1

In [144]:
import pandas as pd

# === Define categorical and numerical columns ===
categorical_cols = ["Gender", "Race", "familysize", "SmokeFam", "milk_consumption", "Hba1c"]
numerical_cols = ["Age", "PIR", "BMI"]  # adjust based on your dataframe

def summarize_dataframe(df, categorical_cols, numerical_cols):
    summary = []

    # Categorical columns: counts and percentages
    for col in categorical_cols:
        counts = df[col].value_counts(dropna=False)
        percents = df[col].value_counts(normalize=True, dropna=False) * 100
        for cat in counts.index:
            summary.append({
                "Variable": col,
                "Category": cat,
                "Count": counts[cat],
                "Percentage": f"{percents[cat]:.2f}%"
            })

    # Numerical columns: median (Q1,Q3)
    for col in numerical_cols:
        median = df[col].median()
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        summary.append({
            "Variable": col,
            "Category": "Median(Q1,Q3)",
            "Count": f"{median:.2f}({q1:.2f},{q3:.2f})",
            "Percentage": ""
        })

    summary_df = pd.DataFrame(summary)
    return summary_df

# === Generate summary tables ===
train_summary = summarize_dataframe(df_final_train, categorical_cols, numerical_cols)
test_summary  = summarize_dataframe(df_final_test, categorical_cols, numerical_cols)

# === Export to CSV if needed ===
train_summary.to_csv("train_summary.csv", index=False)
test_summary.to_csv("test_summary.csv", index=False)

# === Print head to check ===
print("Train Summary:")
print(train_summary.head(20))
print("\nTest Summary:")
print(test_summary.head(20))


Train Summary:
            Variable Category  Count Percentage
0             Gender      2.0  18498     50.81%
1             Gender      1.0  17910     49.19%
2               Race      3.0  16117     44.27%
3               Race      4.0   8289     22.77%
4               Race      1.0   7237     19.88%
5               Race      2.0   2464      6.77%
6               Race      5.0   2301      6.32%
7         familysize      2.0   9342     25.66%
8         familysize      4.0   6751     18.54%
9         familysize      3.0   6551     17.99%
10        familysize      5.0   4793     13.16%
11        familysize      1.0   3915     10.75%
12        familysize      7.0   2644      7.26%
13        familysize      6.0   2412      6.62%
14          SmokeFam        0  28555     78.43%
15          SmokeFam        1   7853     21.57%
16  milk_consumption      3.0  17142     47.08%
17  milk_consumption      2.0   9493     26.07%
18  milk_consumption      1.0   4917     13.51%
19  milk_consumption     